## Gold Payment Summary & Data Quality

 **2 Gold tables**:

- `Gold_Payment_Summary`
- `Gold_Data_Quality`

### Gold_Payment_Summary

Calculates totals by:

- `fund_id`
- `payment_type`
- `source_side`

Internal and external payment values are calculated **separately**, not blended.

**Internal source:** `silver.payment`

**External source:** `silver.external_payment_current`

Only `SETTLED` payments are included, following the same rule used in Notebooks **13 and 14**.

#### Internal Schema

`silver.payment`:

- `payment_id`
- `commitment_id`
- `fund_id`
- `investor_id`
- `company_id`
- `payment_type`
- `amount_usd`
- `status`
- `event_date`
- `entry_benchmark_price`

#### External Schema

`silver.external_payment_current`:

- `payment_thread_id`
- `payment_id`
- `fund_id`
- `internal_fund_id`
- `payment_type`
- `amount`
- `currency`
- `status`
- `event_timestamp`
- `settlement_timestamp`
- `source_system`
- `business_date`
- `thread_event_count`

### Gold_Data_Quality

Consolidates all `log_dq()` records generated across Notebooks **01–14** into one formal Gold deliverable.

Source:

`silver_common.log_dq`

#### DQ Log Schema

- `source_name`
- `business_date`
- `rule_name`
- `records_checked`
- `records_failed`
- `reason_code`
- `created_at`

The table is primarily a **pass-through** of the existing DQ log, with two additional fields:

- `failure_rate`
- `gold_loaded_at`

No major reshaping is required because `log_dq` was already designed as a shared data-quality log.

### Gold Tables Created

| Gold Table | Source | Logic |
|---|---|---|
| `Gold_Payment_Summary` | `silver.payment` + `silver.external_payment_current` | Separate internal/external settled totals |
| `Gold_Data_Quality` | `silver_common.log_dq` | Pass-through + `failure_rate` + load timestamp |


In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F

## A. `Gold_Payment_Summary`

Two independently-computed totals per `(fund_id, payment_type)` - internal ledger and external custodian feed - unioned with a `source_side` tag rather than blended into one number, since (as `13`/`14` already established) the two sides answer different questions and are meant to be compared, not merged. `SETTLED` only, matching the rule used everywhere else in this project.

In [0]:
payment_df = spark.table(silver_table("payment"))
external_payment_current_df = spark.table(silver_table("external_payment_current"))

internal_summary_df = (
    payment_df
    .filter(F.col("status") == "SETTLED")
    .groupBy("fund_id", "payment_type")
    .agg(
        F.sum("amount_usd").alias("total_amount"),
        F.count("*").alias("payment_count")
    )
    .withColumn("source_side", F.lit("INTERNAL"))
)

external_summary_df = (
    external_payment_current_df
    .filter(F.col("status") == "SETTLED")
    .groupBy(F.col("internal_fund_id").alias("fund_id"), "payment_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("payment_count")
    )
    .withColumn("source_side", F.lit("EXTERNAL"))
)

payment_summary_df = (
    internal_summary_df
    .unionByName(external_summary_df)
    .select("fund_id", "payment_type", "source_side", "total_amount", "payment_count")
    .withColumn("gold_loaded_at", F.current_timestamp())
)

write_gold(payment_summary_df, "payment_summary")
print(f"Gold_Payment_Summary row count: {payment_summary_df.count()}")
payment_summary_df.orderBy("fund_id", "payment_type", "source_side").show(50, truncate=False)

Gold_Payment_Summary row count: 18
+--------+------------+-----------+--------------------+-------------+-------------------------+
|fund_id |payment_type|source_side|total_amount        |payment_count|gold_loaded_at           |
+--------+------------+-----------+--------------------+-------------+-------------------------+
|FUND_001|CAPITAL_CALL|INTERNAL   |2.5525862150000002E7|13           |2026-09-23 05:25:25.02535|
|FUND_001|CONTRIBUTION|INTERNAL   |3.3175863380000003E7|18           |2026-09-23 05:25:25.02535|
|FUND_001|DISTRIBUTION|INTERNAL   |435093.35000000003  |5            |2026-09-23 05:25:25.02535|
|FUND_001|INVESTMENT  |INTERNAL   |7814463.41          |9            |2026-09-23 05:25:25.02535|
|FUND_002|CAPITAL_CALL|INTERNAL   |1435030.71          |2            |2026-09-23 05:25:25.02535|
|FUND_002|CONTRIBUTION|INTERNAL   |1.3992916270000003E7|9            |2026-09-23 05:25:25.02535|
|FUND_002|INVESTMENT  |EXTERNAL   |500000.0            |1            |2026-09-23 05:25:25.02

## B. `Gold_Data_Quality`

Direct pass-through of `dq_log` (already a shared, formal log every prior notebook writes to via `log_dq()`) plus one computed convenience column, `failure_rate`, so Power BI does not need a calculated measure just to show the most basic DQ signal.

In [0]:
dq_log_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dq_log")

data_quality_gold_df = (
    dq_log_df
    .withColumn(
        "failure_rate",
        F.when(F.col("records_checked") > 0, F.col("records_failed") / F.col("records_checked"))
         .otherwise(F.lit(None).cast("double"))
    )
    .withColumn("gold_loaded_at", F.current_timestamp())
)

write_gold(data_quality_gold_df, "data_quality")
print(f"Gold_Data_Quality row count: {data_quality_gold_df.count()}")

print("Failure summary by source_name (sanity check):")
(
    data_quality_gold_df
    .groupBy("source_name")
    .agg(
        F.sum("records_checked").alias("total_checked"),
        F.sum("records_failed").alias("total_failed"),
        F.count("*").alias("rule_runs")
    )
    .orderBy("source_name")
    .show(50, truncate=False)
)

data_quality_gold_df.orderBy("source_name", "rule_name").show(50, truncate=False)

Gold_Data_Quality row count: 53
Failure summary by source_name (sanity check):
+-----------------+-------------+------------+---------+
|source_name      |total_checked|total_failed|rule_runs|
+-----------------+-------------+------------+---------+
|cash             |30           |1           |3        |
|commitment       |1320         |264         |4        |
|external_payment |210          |2           |14       |
|fund             |150          |40          |6        |
|investor         |750          |200         |3        |
|market_price     |615          |0           |3        |
|payment          |1844         |16          |10       |
|portfolio_company|600          |120         |4        |
|position         |63           |5           |3        |
|reference        |57           |2           |3        |
+-----------------+-------------+------------+---------+

+-----------------+-------------+-------------------------+---------------+--------------+--------------------------+-----

### Summary

In [0]:
print("=== 16_gold_payments_dq complete ===")
for t in ["payment_summary", "data_quality"]:
    cnt = spark.table(gold_table(t)).count()
    print(f"  {gold_table(t)}: {cnt} rows")

print()
print("All 6 Gold tables now built:")
for t in ["fund_snapshot", "fund_financials", "portfolio_valuation", "reconciliation", "payment_summary", "data_quality"]:
    cnt = spark.table(gold_table(t)).count()
    print(f"  {gold_table(t)}: {cnt} rows")

=== 16_gold_payments_dq complete ===
  dbw_pe_platform.gold.payment_summary: 18 rows
  dbw_pe_platform.gold.data_quality: 53 rows

All 6 Gold tables now built:
  dbw_pe_platform.gold.fund_snapshot: 5 rows
  dbw_pe_platform.gold.fund_financials: 5 rows
  dbw_pe_platform.gold.portfolio_valuation: 30 rows
  dbw_pe_platform.gold.reconciliation: 55 rows
  dbw_pe_platform.gold.payment_summary: 18 rows
  dbw_pe_platform.gold.data_quality: 53 rows
